## Mount to Google Drive and Install Library

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics firebase-admin

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 39.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]


## Detector Engine

In [ ]:
import cv2
import time
import random
import numpy as np
from datetime import datetime
import firebase_admin
from firebase_admin import credentials, db
from ultralytics import YOLO

# 1. Point to Google Drive folder
base_dir = "/content/drive/MyDrive/Thesis_Pothole_Project"
cred_path = f"{base_dir}/Pothole_Simulation/credentials.json"
video_path = f"{base_dir}/pothole_night.mp4"
model_path = f"{base_dir}/Training_Results/Blend(0.6)_Model_AGCWD_YOLOv8n/weights/best.pt"

# 2. Connect to Google Firebase
cred = credentials.Certificate(cred_path)

# Check if Firebase is already initialized to prevent Colab errors if run the cell twice
if not firebase_admin._apps:
    firebase_admin.initialize_app(cred, {'databaseURL': 'https://pothole-detection-175a0-default-rtdb.asia-southeast1.firebasedatabase.app/'})

# 3. Load winning model and video
model = YOLO(model_path)
cap = cv2.VideoCapture(video_path)

# Simulated GPS starting coordinates (Kulim region)
current_lat = 5.3711
current_long = 100.5540

cooldown_time = 5  # Wait 5 seconds before reporting the same pothole again
last_alert_time = 0
frame_count = 0

def apply_agcwd_blend(img, w=0.5, alpha=0.6):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    v_channel = hsv[:, :, 2]
    hist, _ = np.histogram(v_channel.flatten(), 256, [0, 256])
    pdf = hist / np.sum(hist)
    max_pdf, min_pdf = np.max(pdf), np.min(pdf)
    if max_pdf == min_pdf:
        return img
    w_pdf = max_pdf * (((pdf - min_pdf) / (max_pdf - min_pdf)) ** w)
    w_cdf = np.cumsum(w_pdf) / np.sum(w_pdf)
    mapping = np.array([255.0 * ((i / 255.0) ** (1.0 - w_cdf[i])) for i in range(256)], dtype=np.uint8)
    hsv[:, :, 2] = cv2.LUT(v_channel, mapping)
    agcwd_img = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    return cv2.addWeighted(agcwd_img, alpha, img, 1.0 - alpha, 0)

print("🚗 Starting Colab Edge Dashcam Simulation...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("✅ End of video reached. Simulation complete.")
        break

    frame_count += 1

    # Preprocess with CLAHE
    enhanced_frame = apply_agcwd_blend(frame, w=0.5, alpha=0.6)

    # Run YOLO detection
    results = model(enhanced_frame, verbose=False)

    for r in results:
        boxes = r.boxes
        for box in boxes:
            confidence = box.conf[0].item()

            # Trigger alert if confidence is over 70%
            if confidence > 0.70:
                current_time = time.time()

                # Use a cooldown so don't spam the database
                if current_time - last_alert_time > cooldown_time:
                    last_alert_time = current_time

                    # Simulate car moving by slightly changing GPS coordinates
                    current_lat += random.uniform(-0.0005, 0.0005)
                    current_long += random.uniform(-0.0005, 0.0005)

                    # Save the screenshot directly to Google Drive
                    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
                    img_name = f"pothole_{timestamp_str}.jpg"
                    img_save_path = f"{base_dir}/{img_name}"
                    cv2.imwrite(img_save_path, enhanced_frame)

                    # Build the JSON Payload
                    payload = {
                        "lat": round(current_lat, 5),
                        "long": round(current_long, 5),
                        "time": datetime.now().strftime("%I:%M %p"),
                        "image": img_name,
                        "confidence": round(confidence * 100, 1)
                    }

                    # Push to Firebase
                    db.reference('pothole_alerts').push(payload)
                    print(f"🚨 POTHOLE DETECTED (Frame {frame_count})! Data pushed to cloud: {payload}")

cap.release()

🚗 Starting Colab Edge Dashcam Simulation...
🚨 POTHOLE DETECTED (Frame 1586)! Data pushed to cloud: {'lat': 5.37149, 'long': 100.55382, 'time': '05:08 AM', 'image': 'pothole_20260929_050832.jpg', 'confidence': 74.6}
✅ End of video reached. Simulation complete.


## Detector Engine With Image

In [4]:
import cv2
import time
import random
import base64
import numpy as np
from datetime import datetime, timezone, timedelta
import firebase_admin
from firebase_admin import credentials, db
from ultralytics import YOLO

# 1. Point to Google Drive folder
base_dir = "/content/drive/MyDrive/Thesis_Pothole_Project"
cred_path = f"{base_dir}/Pothole_Simulation/credentials.json"
video_path = f"{base_dir}/pothole.mp4"
model_path = f"{base_dir}/Training_Results/Blend(0.6)_Model_AGCWD_YOLOv8n/weights/best.pt"

# 2. Connect to Firebase Realtime Database
if not firebase_admin._apps:
    cred = credentials.Certificate(cred_path)
    firebase_admin.initialize_app(cred, {
        'databaseURL': 'https://pothole-detection-175a0-default-rtdb.asia-southeast1.firebasedatabase.app/'
    })

# 3. Load Model and Video
model = YOLO(model_path)
cap = cv2.VideoCapture(video_path)

current_lat = 5.3711
current_long = 100.5540
cooldown_time = 3
last_alert_time = 0
frame_count = 0

def apply_agcwd_blend(img, w=0.5, alpha=0.6):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    v_channel = hsv[:, :, 2]
    hist, _ = np.histogram(v_channel.flatten(), 256, [0, 256])
    pdf = hist / np.sum(hist)
    max_pdf, min_pdf = np.max(pdf), np.min(pdf)
    if max_pdf == min_pdf:
        return img
    w_pdf = max_pdf * (((pdf - min_pdf) / (max_pdf - min_pdf)) ** w)
    w_cdf = np.cumsum(w_pdf) / np.sum(w_pdf)
    mapping = np.array([255.0 * ((i / 255.0) ** (1.0 - w_cdf[i])) for i in range(256)], dtype=np.uint8)
    hsv[:, :, 2] = cv2.LUT(v_channel, mapping)
    agcwd_img = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    return cv2.addWeighted(agcwd_img, alpha, img, 1.0 - alpha, 0)

def frame_to_base64(img):
    """Resize and convert frame to lightweight Base64 string"""
    resized = cv2.resize(img, (320, 240))  # Resize for fast network transmission
    _, buffer = cv2.imencode('.jpg', resized, [int(cv2.IMWRITE_JPEG_QUALITY), 70])
    return base64.b64encode(buffer).decode('utf-8')

print("🚗 Dashcam Edge Detector running (Base64 Mode)...\n")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("✅ End of video reached. Simulation complete.")
        break

    frame_count += 1

    enhanced_frame = apply_agcwd_blend(frame, w=0.5, alpha=0.6)
    # Add conf=0.70 so r.plot() only draws high-confidence potholes
    results = model(enhanced_frame, conf=0.70, verbose=False, device=0)

    for r in results:
        for box in r.boxes:
            confidence = box.conf[0].item()

            if confidence > 0.70:
                current_time = time.time()

                if current_time - last_alert_time > cooldown_time:
                    last_alert_time = current_time

                    current_lat += random.uniform(-0.0003, 0.0003)
                    current_long += random.uniform(-0.0003, 0.0003)

                    # Draw boxes (with thicker lines for 320x240 popups) and encode
                    annotated_frame = r.plot(line_width=3)
                    img_b64 = frame_to_base64(annotated_frame)

                    payload = {
                        "lat": round(current_lat, 5),
                        "long": round(current_long, 5),
                        "time": datetime.now(timezone(timedelta(hours=8))).strftime("%I:%M:%S %p"),
                        "image_b64": img_b64,
                        "confidence": round(confidence * 100, 1)
                    }

                    db.reference('pothole_alerts').push(payload)
                    print(f"🚨 POTHOLE DETECTED (Frame {frame_count})! Lat: {payload['lat']}, Long: {payload['long']}, Time: {payload['time']}, Conf: {payload['confidence']}%")

cap.release()

🚗 Dashcam Edge Detector running (Base64 Mode)...

🚨 POTHOLE DETECTED (Frame 1)! Lat: 5.37106, Long: 100.5537, Time: 12:15:39 AM, Conf: 77.7%
🚨 POTHOLE DETECTED (Frame 59)! Lat: 5.37089, Long: 100.55375, Time: 12:15:42 AM, Conf: 81.9%
🚨 POTHOLE DETECTED (Frame 153)! Lat: 5.37092, Long: 100.55383, Time: 12:15:45 AM, Conf: 76.3%
🚨 POTHOLE DETECTED (Frame 245)! Lat: 5.37073, Long: 100.55369, Time: 12:15:48 AM, Conf: 85.2%
🚨 POTHOLE DETECTED (Frame 339)! Lat: 5.37098, Long: 100.55361, Time: 12:15:51 AM, Conf: 85.2%
🚨 POTHOLE DETECTED (Frame 436)! Lat: 5.37119, Long: 100.55359, Time: 12:15:55 AM, Conf: 72.3%
🚨 POTHOLE DETECTED (Frame 538)! Lat: 5.3714, Long: 100.55338, Time: 12:15:59 AM, Conf: 71.7%
🚨 POTHOLE DETECTED (Frame 632)! Lat: 5.37168, Long: 100.55315, Time: 12:16:02 AM, Conf: 74.7%
✅ End of video reached. Simulation complete.
